# EMQF Databricks Lakehouse Pipeline

Daily incremental pipeline:
API → RAW Delta → Daily updated datasets → SPARQL enrichment → Dimensions metadata → Quality checks → Final score table.

Run order: top to bottom.

In [0]:
from datetime import datetime
import requests
import gzip
import pandas as pd
import xml.etree.ElementTree as ET
from io import BytesIO

import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, lit, trim, length
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

RUN_DATE = datetime.today().strftime("%Y-%m-%d")

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "emqf"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_PATH = f"{BASE_PATH}/raw"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
EXPORT_PATH = f"{BASE_PATH}/exports"

DEMO_FALLBACK_TO_LATEST_DATE = True  # portfolio/demo mode
print("RUN_DATE:", RUN_DATE)
print("BASE_PATH:", BASE_PATH)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS workspace;
CREATE SCHEMA IF NOT EXISTS workspace.default;
CREATE VOLUME IF NOT EXISTS workspace.default.emqf;

## 1. RAW Layer — Eurostat TOC XML

In [0]:
toc_url = "https://ec.europa.eu/eurostat/api/dissemination/catalogue/toc/xml"

toc_response = requests.get(toc_url, timeout=60)
toc_response.raise_for_status()

raw_df = spark.createDataFrame(
    [(RUN_DATE, toc_url, toc_response.text)],
    ["run_date", "source_url", "xml_payload"]
)

raw_df.write \
    .format("delta") \
    .mode("append") \
    .save(f"{RAW_PATH}/toc_xml")

display(raw_df.select("run_date", "source_url"))

## 2. Parse TOC — Daily updated datasets

In [0]:
root = ET.fromstring(toc_response.content)
ns = "{urn:eu.europa.ec.eurostat.navtree}"

data = {
    "title": [],
    "Code": [],
    "type": [],
    "Location path": [],
    "last_update": [],
    "lastModified": [],
    "dataStart": [],
    "dataEnd": [],
    "values": [],
    "metadata": [],
    "downloadLink": [],
    "Source_institution_EN": []
}

def extract_data(element, parent_path=""):
    title = element.find(f"{ns}title")
    code = element.find(f"{ns}code")
    lastUpdate = element.find(f"{ns}lastUpdate")
    lastModified = element.find(f"{ns}lastModified")
    dataStart = element.find(f"{ns}dataStart")
    dataEnd = element.find(f"{ns}dataEnd")
    values = element.find(f"{ns}values")
    metadata = element.find(f"{ns}metadata")
    downloadLink = element.find(f"{ns}downloadLink")

    title_text = title.text if title is not None else None
    element_type = element.attrib.get("type", None)

    if title_text:
        location_path = f"{parent_path} > {title_text}" if parent_path else title_text
    else:
        location_path = parent_path

    source_institution = None
    for child in element:
        tag_name = child.tag.split("}")[-1]
        if tag_name == "source" and child.attrib.get("language") == "en":
            source_institution = child.text
            break

    data["title"].append(title_text)
    data["Code"].append(code.text if code is not None else None)
    data["type"].append(element_type)
    data["Location path"].append(location_path)
    data["last_update"].append(lastUpdate.text if lastUpdate is not None else None)
    data["lastModified"].append(lastModified.text if lastModified is not None else None)
    data["dataStart"].append(dataStart.text if dataStart is not None else None)
    data["dataEnd"].append(dataEnd.text if dataEnd is not None else None)
    data["values"].append(values.text if values is not None else None)
    data["metadata"].append(metadata.text if metadata is not None else None)
    data["downloadLink"].append(downloadLink.text if downloadLink is not None else None)
    data["Source_institution_EN"].append(source_institution)

    for child in element:
        extract_data(child, location_path)

extract_data(root)

toc_df = pd.DataFrame(data)
toc_df = toc_df.replace(r"^\s*$", pd.NA, regex=True)

cols_to_check = [c for c in toc_df.columns if c != "Location path"]

toc_df = toc_df.dropna(
    subset=cols_to_check,
    how="all"
)

toc_df["type"] = toc_df["type"].fillna("folder")

toc_df["last_update"] = pd.to_datetime(
    toc_df["last_update"],
    errors="coerce",
    dayfirst=True
)

dataset_toc = toc_df[
    toc_df["type"] == "dataset"
].copy()

today = pd.to_datetime(RUN_DATE).date()

filtered_toc = dataset_toc[
    dataset_toc["last_update"].dt.date == today
].copy()

if filtered_toc.empty and DEMO_FALLBACK_TO_LATEST_DATE:
    latest_available_date = dataset_toc["last_update"].dt.date.max()

    print(
        f"No datasets found for RUN_DATE={today}. "
        f"Using latest available date: {latest_available_date}"
    )

    filtered_toc = dataset_toc[
        dataset_toc["last_update"].dt.date == latest_available_date
    ].copy()

else:
    latest_available_date = today

filtered_toc["pipeline_run_date"] = str(today)
filtered_toc["effective_update_date"] = str(latest_available_date)

filtered_toc = filtered_toc.drop_duplicates(
    subset=["Code"]
)

# =========================================================
# Eurostat domain/theme extraction
# =========================================================

DOMAIN_MAPPING = {

    # =====================================================
    # Population and social conditions
    # =====================================================

    "demo": "Population and social conditions",
    "hlth": "Population and social conditions",
    "ilc": "Population and social conditions",
    "edat": "Population and social conditions",

    "lfso": "Population and social conditions",
    "lfsa": "Population and social conditions",
    "lfsq": "Population and social conditions",
    "lfst": "Population and social conditions",
    "lfsi": "Population and social conditions",

    "earn": "Population and social conditions",
    "trng": "Population and social conditions",
    "migr": "Population and social conditions",
    "cens": "Population and social conditions",
    "educ": "Population and social conditions",
    "tus": "Population and social conditions",
    "lc": "Population and social conditions",
    "iss": "Population and social conditions",
    "crim": "Population and social conditions",
    "cult": "Population and social conditions",
    "yth": "Population and social conditions",
    "spr": "Population and social conditions",
    "sprt": "Population and social conditions",
    "gbv": "Population and social conditions",
    "hsw": "Population and social conditions",
    "hbs": "Population and social conditions",
    "med": "Population and social conditions",
    "qoe": "Population and social conditions",

    "aact": "Population and social conditions",
    "bd": "Population and social conditions",
    "gba": "Population and social conditions",
    "urt": "Population and social conditions",

    # =====================================================
    # Economy and finance
    # =====================================================

    "nama": "Economy and finance",
    "namq": "Economy and finance",
    "naio": "Economy and finance",
    "prc": "Economy and finance",
    "gov": "Economy and finance",
    "ei": "Economy and finance",

    "bop": "Economy and finance",
    "bs": "Economy and finance",
    "proj": "Economy and finance",
    "met": "Economy and finance",
    "irt": "Economy and finance",
    "aei": "Economy and finance",
    "une": "Economy and finance",
    "ert": "Economy and finance",
    "dt": "Economy and finance",
    "gvc": "Economy and finance",

    "nasa": "Economy and finance",
    "nasq": "Economy and finance",
    "naidsa": "Economy and finance",
    "naidsq": "Economy and finance",
    "naida": "Economy and finance",
    "naidq": "Economy and finance",

    "cei": "Economy and finance",
    "egi": "Economy and finance",
    "egr": "Economy and finance",
    "fobs": "Economy and finance",
    "mips": "Economy and finance",

    # =====================================================
    # Science, technology and digital society
    # =====================================================

    "rd": "Science, technology and digital society",
    "tin": "Science, technology and digital society",
    "isoc": "Science, technology and digital society",
    "inn": "Science, technology and digital society",
    "htec": "Science, technology and digital society",
    "hrst": "Science, technology and digital society",
    "pat": "Science, technology and digital society",
    "icw": "Science, technology and digital society",

    # =====================================================
    # Environment and energy
    # =====================================================

    "env": "Environment and energy",
    "nrg": "Environment and energy",
    "ef": "Environment and energy",
    "enps": "Environment and energy",
    "enpe": "Environment and energy",
    "cli": "Environment and energy",

    # =====================================================
    # Transport
    # =====================================================

    "tran": "Transport",
    "rail": "Transport",
    "road": "Transport",
    "avia": "Transport",
    "mar": "Transport",
    "iww": "Transport",
    "pipe": "Transport",

    # =====================================================
    # Industry, trade and services
    # =====================================================

    "tour": "Industry, trade and services",
    "sbs": "Industry, trade and services",
    "sts": "Industry, trade and services",

    "fats": "Industry, trade and services",
    "jvs": "Industry, trade and services",
    "org": "Industry, trade and services",

    "tipsho10": "Industry, trade and services",
    "tipsho20": "Industry, trade and services",
    "tipsho30": "Industry, trade and services",
    "tipsho40": "Industry, trade and services",
    "tipsho60": "Industry, trade and services",

    "tipsbd10": "Industry, trade and services",
    "tipsbd20": "Industry, trade and services",
    "tipsbd30": "Industry, trade and services",
    "tipsbd40": "Industry, trade and services",

    "tipser10": "Industry, trade and services",
    "tipser20": "Industry, trade and services",

    "tipslm50": "Industry, trade and services",

    # =====================================================
    # International trade
    # =====================================================

    "ext": "International trade",

    # =====================================================
    # Agriculture, fisheries and forestry
    # =====================================================

    "agr": "Agriculture, fisheries and forestry",
    "fish": "Agriculture, fisheries and forestry",
    "for": "Agriculture, fisheries and forestry",

    "apro": "Agriculture, fisheries and forestry",
    "apri": "Agriculture, fisheries and forestry",
    "orch": "Agriculture, fisheries and forestry",
    "vit": "Agriculture, fisheries and forestry",
    "acf": "Agriculture, fisheries and forestry",

    # =====================================================
    # General and regional statistics
    # =====================================================

    "reg": "General and regional statistics",
    "urb": "General and regional statistics",
    "lan": "General and regional statistics",

    # =====================================================
    # Sustainable Development Goals
    # =====================================================

    "sdg": "Sustainable development goals"
}

def extract_domain_acronym(dataset_code):
    if pd.isna(dataset_code):
        return None
    return str(dataset_code).split("_")[0].lower()

filtered_toc["domain_acronym"] = filtered_toc["Code"].apply(extract_domain_acronym)
filtered_toc["top_theme"] = filtered_toc["domain_acronym"].map(DOMAIN_MAPPING)

filtered_toc_small = filtered_toc[
    [
        "Code",
        "title",
        "type",
        "Location path",
        "domain_acronym",
        "top_theme",
        "last_update",
        "lastModified",
        "dataStart",
        "dataEnd",
        "metadata",
        "downloadLink",
        "Source_institution_EN",
        "pipeline_run_date",
        "effective_update_date"
    ]
].copy()

filtered_toc_small = filtered_toc_small.rename(columns={
    "Code": "dataset_code",
    "Location path": "location_path",
    "lastModified": "last_modified",
    "dataStart": "data_start",
    "dataEnd": "data_end",
    "downloadLink": "download_link",
    "Source_institution_EN": "source_institution_en"
})

for col in filtered_toc_small.columns:
    filtered_toc_small[col] = filtered_toc_small[col].astype(str)

daily_schema = StructType([
    StructField("dataset_code", StringType(), True),
    StructField("title", StringType(), True),
    StructField("type", StringType(), True),
    StructField("location_path", StringType(), True),
    StructField("domain_acronym", StringType(), True),
    StructField("top_theme", StringType(), True),
    StructField("last_update", StringType(), True),
    StructField("last_modified", StringType(), True),
    StructField("data_start", StringType(), True),
    StructField("data_end", StringType(), True),
    StructField("metadata", StringType(), True),
    StructField("download_link", StringType(), True),
    StructField("source_institution_en", StringType(), True),
    StructField("pipeline_run_date", StringType(), True),
    StructField("effective_update_date", StringType(), True)
])

daily_datasets_df = spark.createDataFrame(
    filtered_toc_small,
    schema=daily_schema
)

daily_datasets_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_daily_updated_datasets")

print(f"Datasets selected for this run: {daily_datasets_df.count()}")

display(daily_datasets_df)

## 3. SPARQL catalogue enrichment

In [0]:
endpoint = "https://data.europa.eu/sparql"

query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX adms: <http://www.w3.org/ns/adms#>

SELECT DISTINCT ?dataset ?identifier ?identifierDOI ?landingPage
WHERE {
  <http://data.europa.eu/88u/catalogue/estat> a dcat:Catalog ;
      dcat:dataset ?dataset .

  ?dataset dct:identifier ?identifier .

  OPTIONAL {
    ?dataset adms:identifier ?identifierDOI .
    FILTER (STRSTARTS(STR(?identifierDOI), "https://doi.org/"))
  }

  OPTIONAL {
    ?dataset dcat:landingPage ?landingPage .
  }
}
"""

try:
    sparql_response = requests.get(
        endpoint,
        params={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=60
    )
    sparql_response.raise_for_status()
    results = sparql_response.json()

    sparql_df = pd.json_normalize(results["results"]["bindings"])

    if not sparql_df.empty:
        sparql_df.columns = [c.replace(".value", "") for c in sparql_df.columns]
        sparql_df = sparql_df.drop(
            columns=["dataset.type", "identifier.type", "identifierDOI.type", "landingPage.type"],
            errors="ignore"
        )
    else:
        sparql_df = pd.DataFrame(columns=["dataset", "identifier", "identifierDOI", "landingPage"])

except Exception as e:
    print(f"Failed to process SPARQL query results: {e}")
    sparql_df = pd.DataFrame(columns=["dataset", "identifier", "identifierDOI", "landingPage"])

for c in ["dataset", "identifier", "identifierDOI", "landingPage"]:
    if c not in sparql_df.columns:
        sparql_df[c] = None

sparql_df = sparql_df[["dataset", "identifier", "identifierDOI", "landingPage"]].astype(str)

sparql_schema = StructType([
    StructField("dataset", StringType(), True),
    StructField("identifier", StringType(), True),
    StructField("identifierDOI", StringType(), True),
    StructField("landingPage", StringType(), True)
])

sparql_spark_df = spark.createDataFrame(sparql_df, schema=sparql_schema)

sparql_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_sparql_catalogue")

display(sparql_spark_df.limit(20))

## 4. Metabase dimensions — filtered to daily datasets

In [0]:
metabase_url = "https://ec.europa.eu/eurostat/api/dissemination/catalogue/metabase.txt.gz"

response = requests.get(metabase_url, timeout=60)
response.raise_for_status()

with gzip.open(BytesIO(response.content)) as f:
    metabase_data = f.read()

dimensions = pd.read_csv(
    BytesIO(metabase_data),
    sep="\t",
    names=["Dataset_Code", "CODELIST_CODE", "CODE"],
    dtype=str
)

daily_codes = [r["dataset_code"] for r in spark.table("workspace.default.emqf_daily_updated_datasets").select("dataset_code").distinct().collect()]

dimensions = dimensions[dimensions["Dataset_Code"].isin(daily_codes)].copy()
dimensions = dimensions.astype(str)

dimensions_schema = StructType([
    StructField("Dataset_Code", StringType(), True),
    StructField("CODELIST_CODE", StringType(), True),
    StructField("CODE", StringType(), True)
])

dimensions_spark_df = spark.createDataFrame(dimensions, schema=dimensions_schema)

dimensions_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_metabase_dimensions")

print(f"Dimension rows for daily datasets: {dimensions_spark_df.count()}")
display(dimensions_spark_df.limit(50))

## 5. REFCL / codelist metadata — only daily codelists

In [0]:
dimensions = spark.table("workspace.default.emqf_metabase_dimensions").toPandas()

distinct_codelists = (
    dimensions["CODELIST_CODE"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

def read_public_codelist_metadata(codelist_code):
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
        f"structure/codelist/ESTAT/{codelist_code}"
        "?format=TSV&formatVersion=2.0"
    )

    try:
        df = pd.read_csv(url, sep="\t")
        df.columns = df.columns.str.strip()

        title_col = next(
            (c for c in df.columns if c.lower() in ["codelist label", "name", "label"]),
            None
        )

        standard_col = next(
            (c for c in df.columns if c.lower() == "standard code"),
            None
        )

        title_en = (
            df[title_col].dropna().iloc[0]
            if title_col and df[title_col].notna().any()
            else None
        )

        if standard_col:
            values = df[standard_col].dropna().astype(str).str.strip().unique()
            if len(values) == 0:
                standard_code_sc = "empty"
            elif "Y" in values:
                standard_code_sc = "Y"
            elif "C" in values:
                standard_code_sc = "C"
            elif "M" in values:
                standard_code_sc = "M"
            elif "O" in values:
                standard_code_sc = "O"
            elif "N" in values:
                standard_code_sc = "N"
            else:
                standard_code_sc = values[0]
        else:
            standard_code_sc = "empty"

        return {
            "OBJECT_TYPE": "D",
            "CODELIST_CODE": codelist_code,
            "TITLE_EN": title_en,
            "STANDARD_CODE_SC": standard_code_sc,
            "DERIVED_CL": ""
        }

    except Exception as e:
        print(f"Failed to fetch codelist {codelist_code}: {e}")
        return {
            "OBJECT_TYPE": "D",
            "CODELIST_CODE": codelist_code,
            "TITLE_EN": None,
            "STANDARD_CODE_SC": "empty",
            "DERIVED_CL": ""
        }

def get_derived_cl(codelist_code):
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
        f"structure/codelist/ESTAT/{codelist_code}?format=SDMX-JSON"
    )

    try:
        response = requests.get(url, timeout=30)
        if response.status_code != 200:
            return ""

        data = response.json()
        matches = []

        def search_obj(obj):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if any(word in k.lower() for word in ["derived", "parent", "source", "base"]):
                        if isinstance(v, str):
                            matches.append(v)
                    search_obj(v)
            elif isinstance(obj, list):
                for item in obj:
                    search_obj(item)

        search_obj(data)

        derived_cls = list(set([
            x for x in matches
            if isinstance(x, str) and x.startswith("CL_")
        ]))

        return ", ".join(derived_cls)

    except Exception:
        return ""

refcl = pd.DataFrame([read_public_codelist_metadata(cl) for cl in distinct_codelists])

if refcl.empty:
    refcl = pd.DataFrame(columns=["OBJECT_TYPE", "CODELIST_CODE", "TITLE_EN", "STANDARD_CODE_SC", "DERIVED_CL"])

refcl["DERIVED_CL"] = refcl["CODELIST_CODE"].astype(str).apply(get_derived_cl)
refcl = refcl.astype(str)

refcl_schema = StructType([
    StructField("OBJECT_TYPE", StringType(), True),
    StructField("CODELIST_CODE", StringType(), True),
    StructField("TITLE_EN", StringType(), True),
    StructField("STANDARD_CODE_SC", StringType(), True),
    StructField("DERIVED_CL", StringType(), True)
])

refcl_spark_df = spark.createDataFrame(refcl, schema=refcl_schema)

refcl_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_refcl")

display(refcl_spark_df)

## 6. Enriched dimensions metadata

In [0]:
dimensions = spark.table("workspace.default.emqf_metabase_dimensions").toPandas()
refcl = spark.table("workspace.default.emqf_refcl").toPandas()

dimensions = dimensions.merge(
    refcl[["CODELIST_CODE", "STANDARD_CODE_SC", "DERIVED_CL"]],
    on="CODELIST_CODE",
    how="left"
)

dimensions["STANDARD_CODE_SC"] = dimensions["STANDARD_CODE_SC"].fillna("empty")
dimensions["DERIVED_CL"] = dimensions["DERIVED_CL"].fillna("")
dimensions = dimensions.astype(str)

dimensions_enriched_schema = StructType([
    StructField("Dataset_Code", StringType(), True),
    StructField("CODELIST_CODE", StringType(), True),
    StructField("CODE", StringType(), True),
    StructField("STANDARD_CODE_SC", StringType(), True),
    StructField("DERIVED_CL", StringType(), True)
])

dimensions_enriched_spark_df = spark.createDataFrame(dimensions, schema=dimensions_enriched_schema)

dimensions_enriched_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_dimensions_metadata")

display(dimensions_enriched_spark_df.limit(50))

## 7. Join daily datasets with SPARQL

In [0]:
daily_datasets_df = spark.table("workspace.default.emqf_daily_updated_datasets")
sparql_spark_df = spark.table("workspace.default.emqf_sparql_catalogue")

merged_df = daily_datasets_df.join(
    sparql_spark_df,
    daily_datasets_df.dataset_code == sparql_spark_df.identifier,
    "left"
).drop(sparql_spark_df.identifier)

merged_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_daily_datasets_enriched")

display(merged_df)

## 6. Abbreviations

In [0]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

url = "https://ec.europa.eu/eurostat/esa2010/chapter/view/27/"

response = requests.get(url, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

text = soup.get_text("\n", strip=True)

lines = [
    re.sub(r"\s+", " ", line.strip())
    for line in text.split("\n")
    if line.strip()
]

abbr_positions = [
    i for i, line in enumerate(lines)
    if line == "List of abbreviations and acronyms"
]

if len(abbr_positions) < 2:
    raise ValueError("Could not find the second 'List of abbreviations and acronyms' section.")

start_idx = abbr_positions[1]

end_idx = next(
    i for i, line in enumerate(lines[start_idx:], start=start_idx)
    if line == "Glossary"
)

abbr_lines = lines[start_idx + 1:end_idx]

rows = []

for i in range(0, len(abbr_lines) - 1, 2):
    abbreviation = abbr_lines[i]
    title = abbr_lines[i + 1]

    rows.append({
        "abbreviation": abbreviation,
        "abbreviation_title": title
    })

abbreviation_df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .reset_index(drop=True)
)

abbreviation_df = abbreviation_df.astype(str)

abbreviation_spark_df = spark.createDataFrame(abbreviation_df)

abbreviation_spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_esa2010_abbreviations")

display(abbreviation_spark_df)

## 8. Quality checks — build once, write once

In [0]:
import pandas as pd
import numpy as np
import re
import requests

from pyspark.sql.functions import col, when, lit, trim, length
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, BooleanType
)

checks_df = spark.table("workspace.default.emqf_daily_datasets_enriched")

# =========================================================
# Basic checks: metadata / DOI / OP / title / historical
# =========================================================

checks_df = checks_df.withColumn(
    "metadata_file_existence_esms_score",
    when(col("metadata").isNotNull() & (~col("metadata").isin("None", "nan", "")), lit(100.0)).otherwise(lit(0.0))
).withColumn(
    "metadata_file_existence_esms_error",
    when(col("metadata_file_existence_esms_score") == 0, lit("Missing ESMS metadata file")).otherwise(lit(None).cast("string"))
).withColumn(
    "doi_verification_score",
    when(
        col("identifierDOI").isNotNull() &
        (~col("identifierDOI").isin("None", "nan", "")) &
        (col("identifierDOI").startswith("https://doi.org/")),
        lit(100.0)
    ).otherwise(lit(0.0))
).withColumn(
    "doi_verification_error",
    when(col("doi_verification_score") == 0, lit("Missing DOI")).otherwise(lit(None).cast("string"))
).withColumn(
    "op_dataset_availability_score",
    when(col("landingPage").isNotNull() & (~col("landingPage").isin("None", "nan", "")), lit(100.0)).otherwise(lit(0.0))
).withColumn(
    "op_dataset_availability_error",
    when(col("op_dataset_availability_score") == 0, lit("Missing OP landing page")).otherwise(lit(None).cast("string"))
)

dataset_codes = [
    r["dataset_code"]
    for r in checks_df.select("dataset_code").distinct().collect()
]

# =========================================================
# Data Browser Verification
# =========================================================

browser_rows = []

for dataset_code in dataset_codes:
    try:
        if dataset_code is None or dataset_code in ["None", "nan", ""]:
            score = 0.0
        else:
            browser_url = f"https://ec.europa.eu/eurostat/databrowser/view/{dataset_code}/default/table?lang=en"
            browser_response = requests.get(browser_url, timeout=15)
            score = 100.0 if browser_response.status_code == 200 else 0.0
    except Exception:
        score = 0.0

    browser_rows.append({
        "browser_dataset_code": dataset_code,
        "data_browser_verification_score": score,
        "data_browser_verification_error": None if score == 100.0 else "Dataset not available in Eurostat Data Browser"
    })

browser_schema = StructType([
    StructField("browser_dataset_code", StringType(), True),
    StructField("data_browser_verification_score", DoubleType(), True),
    StructField("data_browser_verification_error", StringType(), True)
])

browser_df = spark.createDataFrame(pd.DataFrame(browser_rows), schema=browser_schema)

checks_df = checks_df.join(
    browser_df,
    checks_df.dataset_code == browser_df.browser_dataset_code,
    "left"
).drop("browser_dataset_code")

# =========================================================
# Source of Data Validation
# =========================================================

daily_df = spark.table("workspace.default.emqf_daily_updated_datasets")
abbreviations_df = spark.table("workspace.default.emqf_esa2010_abbreviations")

source_pd = daily_df.toPandas()
abbreviations_pd = abbreviations_df.toPandas()

source_pd = source_pd.rename(columns={
    "dataset_code": "Code",
    "title": "Title EN",
    "source_institution_en": "Source_institution_EN",
    "dataStart": "dataStart",
    "dataEnd": "dataEnd"
})

def extract_bracket_contents(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    matches = re.findall(r"\((.*?)\)", text)
    return ", ".join(matches)

source_pd["Extracted_Text"] = source_pd["Source_institution_EN"].apply(extract_bracket_contents)

source_pd["Source_Eurostat"] = source_pd["Source_institution_EN"].apply(
    lambda x: "Y" if pd.notna(x) and "Eurostat" in str(x) else "N"
)

def check_order(row):
    source = str(row["Source_institution_EN"])

    if row["Source_Eurostat"] == "N":
        return "Y"
    elif row["Source_Eurostat"] == "Y" and source.startswith("Eurostat"):
        return "Y"
    else:
        return "N"

source_pd["order_check"] = source_pd.apply(check_order, axis=1)

abbreviations_set = set(
    abbreviations_pd["abbreviation"]
    .dropna()
    .astype(str)
    .str.strip()
)

roman_numerals_set = {
    "I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X",
    "XI", "XII", "XIII", "XIV", "XV", "XVI", "XVII", "XVIII", "XIX", "XX"
}

def extract_acronyms_and_explanations(text):
    if pd.isna(text):
        return [], {}

    text = str(text)
    pattern = re.compile(r"(?:(\b[A-Za-z ]+\b) \(([A-Z]{2,})\))|(\b[A-Z]{2,}\b)")
    matches = pattern.findall(text)

    acronyms = []
    explained = {}

    for match in matches:
        if match[1]:
            explained[match[1]] = match[0]
            acronyms.append(match[1])
        elif match[2]:
            acronyms.append(match[2])

    return acronyms, explained

def check_acronyms(title):
    acronyms, explained = extract_acronyms_and_explanations(title)

    passed = []
    failed = []

    for acronym in acronyms:
        if acronym in abbreviations_set or acronym in explained or acronym in roman_numerals_set:
            passed.append(acronym)
        else:
            failed.append(acronym)

    return len(failed) == 0, ", ".join(passed), ", ".join(failed)

source_pd[[
    "source_acronym_check",
    "source_acronyms_passed",
    "source_acronyms_failed"
]] = source_pd["Title EN"].apply(
    lambda title: pd.Series(check_acronyms(title))
)

source_pd["source_abbreviation_check"] = source_pd["Source_institution_EN"].apply(
    lambda x: "N/A" if pd.isna(x) or str(x).strip() == "" else "Y"
)

condition_1 = source_pd["source_abbreviation_check"].isin(["Y", "N/A"])
condition_2 = source_pd["order_check"] == "Y"

source_pd["source_of_data_validation_score"] = (
    condition_1.astype(int) + condition_2.astype(int)
) * 50.0

def generate_source_error(row):
    errors = []

    if row["order_check"] != "Y":
        errors.append('Source of data does not start with "Eurostat" when required')

    if row["source_abbreviation_check"] not in ["Y", "N/A"]:
        errors.append("Source of data is not recognized")

    if row["source_acronym_check"] is False:
        errors.append("Unrecognized acronym(s) in dataset title")

    if len(errors) == 0:
        return None

    return "Source of Data Validation [" + " and ".join(errors) + "]"

source_pd["source_of_data_validation_error"] = source_pd.apply(generate_source_error, axis=1)

source_results_pd = source_pd[[
    "Code",
    "Source_Eurostat",
    "order_check",
    "source_abbreviation_check",
    "source_acronym_check",
    "source_acronyms_passed",
    "source_acronyms_failed",
    "source_of_data_validation_score",
    "source_of_data_validation_error"
]].copy()

source_results_pd = source_results_pd.rename(columns={
    "Code": "source_dataset_code",
    "Source_Eurostat": "source_eurostat",
    "order_check": "source_order_check"
})

source_results_pd = source_results_pd.astype(str)
source_results_pd["source_of_data_validation_score"] = (
    source_results_pd["source_of_data_validation_score"].astype(float)
)

source_schema = StructType([
    StructField("source_dataset_code", StringType(), True),
    StructField("source_eurostat", StringType(), True),
    StructField("source_order_check", StringType(), True),
    StructField("source_abbreviation_check", StringType(), True),
    StructField("source_acronym_check", StringType(), True),
    StructField("source_acronyms_passed", StringType(), True),
    StructField("source_acronyms_failed", StringType(), True),
    StructField("source_of_data_validation_score", DoubleType(), True),
    StructField("source_of_data_validation_error", StringType(), True)
])

source_results_df = spark.createDataFrame(source_results_pd, schema=source_schema)

checks_df = checks_df.join(
    source_results_df,
    checks_df.dataset_code == source_results_df.source_dataset_code,
    "left"
).drop("source_dataset_code")

# =========================================================
# Dataset Title Standards Verification
# =========================================================

# source_pd already contains:
# Code, Title EN, Source_institution_EN, source_acronym_check, etc.

dimensions_for_title = spark.table("workspace.default.emqf_dimensions_metadata").toPandas()

source_results_df = source_pd.copy()

# Product type fallbacks
if "Reference Product Type" not in source_results_df.columns:
    source_results_df["Reference Product Type"] = source_results_df.get("type", "Dataset").astype(str).str.title()

if "Dissemination Product Type" not in source_results_df.columns:
    source_results_df["Dissemination Product Type"] = source_results_df.get("type", "dataset").astype(str).str.upper()

if "Location path" not in source_results_df.columns and "location_path" in source_results_df.columns:
    source_results_df["Location path"] = source_results_df["location_path"]

# Section
conditions = [
    source_results_df["Location path"].fillna("").str.contains("Tables on EU policy", case=False, regex=False),
    source_results_df["Location path"].fillna("").str.contains("Cross cutting topics", case=False, regex=False)
]

choices = ["eu", "cc"]

source_results_df["Section"] = np.select(
    conditions,
    choices,
    default=source_results_df["Reference Product Type"]
)

# BY breakdown
def check_by_and_next_word(title):
    title = "" if pd.isna(title) else str(title)
    words = title.split()
    by_indices = [i for i, word in enumerate(words) if word.lower() == "by"]

    if len(by_indices) != 1:
        return len(by_indices) == 0, None

    by_index = by_indices[0]

    if by_index + 1 < len(words):
        return True, words[by_index + 1]

    return True, None

source_results_df[["by_breakdown_check", "word_after_by"]] = source_results_df["Title EN"].apply(
    lambda title: check_by_and_next_word(title)
).apply(pd.Series)

# NACE
def check_nace_condition(title):
    title = "" if pd.isna(title) else str(title)

    if "NACE" not in title:
        return True

    words = title.split()

    for i, word in enumerate(words):
        if word == "NACE":
            return i + 1 < len(words) and words[i + 1] == "Rev."

    return True

source_results_df["nace_breakdown_check"] = source_results_df["Title EN"].apply(check_nace_condition)

# Region
source_results_df["region_breakdown_check"] = ~source_results_df["Title EN"].str.contains(
    "regions",
    case=False,
    na=False
)

source_results_df["breakdown_check"] = source_results_df[
    [
        "nace_breakdown_check",
        "region_breakdown_check",
        "by_breakdown_check"
    ]
].all(axis=1)

# Length
source_results_df["length_check"] = source_results_df["Title EN"].apply(
    lambda title: len(str(title)) <= 149
)

# Separators
def check_separators(title):
    title = "" if pd.isna(title) else str(title)

    matches = re.findall(r"\((.*?)\)", title)

    if len(matches) <= 1:
        return True

    has_valid_year_or_range = False
    has_all_caps = False
    has_equals_in_parentheses = False

    for match in matches:
        if re.search(r"\b\d{4}-\d{4}\b", match) or re.search(r"\b\d{4}\b", match):
            has_valid_year_or_range = True
        elif match.isupper():
            has_all_caps = True

        if "=" in match:
            has_equals_in_parentheses = True

    return has_valid_year_or_range or has_all_caps or has_equals_in_parentheses

source_results_df["separators_check"] = source_results_df["Title EN"].apply(check_separators)

# First letter
def is_first_letter_capitalized_or_number(title):
    title = "" if pd.isna(title) else str(title).strip()
    return title[0].isupper() or title[0].isdigit() if title else False

source_results_df["letters_check"] = source_results_df["Title EN"].apply(
    is_first_letter_capitalized_or_number
)

# Use acronym check from Source validation
if "source_acronym_check" in source_results_df.columns:
    source_results_df["acronym_check"] = source_results_df["source_acronym_check"]
else:
    source_results_df["acronym_check"] = True

source_results_df["acronym_check"] = source_results_df.apply(
    lambda row: True if "COVID-19" in str(row["Title EN"]) and not row["acronym_check"]
    else False if "Covid-19" in str(row["Title EN"])
    else row["acronym_check"],
    axis=1
)

# Spaces
source_results_df["spaces_check"] = ~source_results_df["Title EN"].str.contains("  ", na=False)

# Index check
def check_units(title):
    title = "" if pd.isna(title) else str(title)

    if re.search(r"\(.*\d+.*=.*\d+.*\)", title):
        matches = re.findall(r"\(([^)]+)\)", title)

        for match in matches:
            if re.search(r"\b\d+\s=\s\d+\b", match):
                return True

        return False

    return True

source_results_df["index_check"] = source_results_df["Title EN"].apply(check_units)

# Periodicity
terms = [
    "weekly data",
    "quarterly data",
    "monthly data",
    "daily data",
    "annual data",
    "quarterly and annual data",
    "bi-annual data"
]

def check_prefix(title):
    title = "" if pd.isna(title) else str(title)

    term_found = False

    for term in terms:
        pattern = rf" - {re.escape(term)}(?:[\s,]|$)"

        if re.search(pattern, title):
            return True

        if term in title:
            term_found = True

    if not term_found:
        return True

    return False

source_results_df["periodicity_check"] = source_results_df["Title EN"].apply(check_prefix)

# Indicator
source_results_df["total_indicator_check"] = ~source_results_df["Title EN"].str.strip().str.lower().str.startswith("total")
source_results_df["people_indicator_check"] = ~source_results_df["Title EN"].str.contains("people", case=False, na=False)

def check_parentheses_condition(title):
    title = "" if pd.isna(title) else str(title)

    if "greater than" in title or "less than" in title:
        parenthesis_content = re.findall(r"\((.*?)\)", title)

        for content in parenthesis_content:
            if "greater than" in content or "less than" in content:
                return True

        return False

    return True

source_results_df["observational_indicator_check"] = source_results_df["Title EN"].apply(
    check_parentheses_condition
)

source_results_df["indicator_check"] = source_results_df[
    [
        "total_indicator_check",
        "people_indicator_check",
        "observational_indicator_check"
    ]
].all(axis=1)

# Units
source_results_df["breakdown_units_check"] = ~source_results_df["Title EN"].str.contains(
    r"\b(?:breakdown|breakdowns)\b",
    case=False,
    na=False
)

patterns_to_flag = [
    r"\b100 000\b",
    r"\b1 000\b",
    r"\b1000\b",
    r"\b100\b",
    r"\b10\b"
]

source_results_df["numbers_units_check"] = source_results_df["Title EN"].apply(
    lambda x: (
        True if "100" in str(x) and "=" in str(x) and "(" in str(x) and ")" in str(x) else
        True if str(x).strip() == "= 100" else
        True if "*10" in str(x) else
        False if any(pd.Series(patterns_to_flag).apply(
            lambda p: bool(pd.Series([str(x)]).str.contains(p, regex=True).any())
        )) else
        True
    )
)

def check_percentage(title):
    title = "" if pd.isna(title) else str(title).lower()

    percentage_phrases = [
        "percentage point",
        "percentage points",
        "percentage change",
        "percentage changes"
    ]

    for phrase in percentage_phrases:
        if phrase in title:
            return True

    if "percentage" in title:
        if title.index("percentage") == 0 or " by percentage " in title:
            return True

    if " - % " in title:
        return True

    return "percentage" not in title and "%" not in title

source_results_df["percentage_units_check"] = source_results_df["Title EN"].apply(check_percentage)

source_results_df["units_check"] = source_results_df[
    [
        "breakdown_units_check",
        "numbers_units_check",
        "percentage_units_check"
    ]
].all(axis=1)

# Uniqueness check
dimensions_for_title["Dataset_Code"] = dimensions_for_title["Dataset_Code"].astype(str).str.strip()
dimensions_for_title["CODELIST_CODE"] = dimensions_for_title["CODELIST_CODE"].astype(str).str.strip().str.upper()
dimensions_for_title["CODE"] = dimensions_for_title["CODE"].astype(str).str.strip().str.upper()

source_results_df = source_results_df.drop_duplicates(subset=["Code", "Section"])

dataset_df = source_results_df[source_results_df["Section"] == "Dataset"].drop_duplicates(
    subset="Code",
    keep="first"
)

dataset_uniq = dataset_df[dataset_df.duplicated(subset="Title EN", keep=False)].copy()
dataset_uniq["Is_Match"] = False

def compare_dimensions(df_with_pairs, dimensions, code_col_1, code_col_2):
    def compare_codes(row):
        code1 = row[code_col_1]
        code2 = row[code_col_2]

        df1 = dimensions[dimensions["Dataset_Code"] == code1]
        df2 = dimensions[dimensions["Dataset_Code"] == code2]

        if df1.empty or df2.empty:
            return True

        df1_filtered = df1[
            ~df1["CODELIST_CODE"].isin(["GEO", "TIME"])
        ][["CODELIST_CODE", "CODE"]].drop_duplicates()

        df2_filtered = df2[
            ~df2["CODELIST_CODE"].isin(["GEO", "TIME"])
        ][["CODELIST_CODE", "CODE"]].drop_duplicates()

        df1_sorted = df1_filtered.sort_values(by=["CODELIST_CODE", "CODE"]).reset_index(drop=True)
        df2_sorted = df2_filtered.sort_values(by=["CODELIST_CODE", "CODE"]).reset_index(drop=True)

        return df1_sorted.equals(df2_sorted)

    if df_with_pairs.empty:
        df_with_pairs["Is_Match"] = []
        return df_with_pairs

    df_with_pairs["Is_Match"] = df_with_pairs.apply(compare_codes, axis=1)
    return df_with_pairs

def create_comparison_df(source_df, section_label, suffix):
    df_section = source_df[source_df["Section"] == section_label]
    merged = df_section.merge(source_df, on="Title EN", suffixes=(f"_{suffix}", "_other"))

    condition = (
        (merged[f"Code_{suffix}"] != merged["Code_other"]) &
        (merged["Section_other"] != section_label)
    )

    filtered = merged[condition]

    if section_label == "eu":
        filtered = filtered[
            ~((filtered["Section_other"] == "eu") & (filtered[f"Section_{suffix}"] == "eu"))
        ]

    if filtered.empty:
        return pd.DataFrame(columns=[
            f"Code_{suffix}",
            "Code_other",
            "Title EN",
            f"Section_{suffix}",
            f"Reference Product Type_{suffix}",
            "Section_other",
            "Reference Product Type_other"
        ])

    filtered = filtered[
        [
            f"Code_{suffix}",
            "Title EN",
            f"Section_{suffix}",
            f"Reference Product Type_{suffix}",
            "Code_other",
            "Section_other",
            "Reference Product Type_other"
        ]
    ]

    result = filtered.groupby([f"Code_{suffix}", "Code_other"]).agg({
        "Title EN": "first",
        f"Section_{suffix}": "first",
        f"Reference Product Type_{suffix}": "first",
        "Section_other": lambda x: ", ".join(sorted(set(x))),
        "Reference Product Type_other": "first"
    }).reset_index()

    return result

eu_uniq = compare_dimensions(
    create_comparison_df(source_results_df, "eu", "eu"),
    dimensions_for_title,
    "Code_eu",
    "Code_other"
)

cc_uniq = compare_dimensions(
    create_comparison_df(source_results_df, "cc", "cc"),
    dimensions_for_title,
    "Code_cc",
    "Code_other"
)

derived_uniq = compare_dimensions(
    create_comparison_df(source_results_df, "Derived dataset", "derived"),
    dimensions_for_title,
    "Code_derived",
    "Code_other"
)

map_dataset = dict(zip(dataset_uniq["Code"], dataset_uniq["Is_Match"]))
map_derived = dict(zip(derived_uniq.get("Code_derived", []), derived_uniq.get("Is_Match", [])))
map_cc = dict(zip(cc_uniq.get("Code_cc", []), cc_uniq.get("Is_Match", [])))
map_eu = dict(zip(eu_uniq.get("Code_eu", []), eu_uniq.get("Is_Match", [])))

combined_map = {
    **map_derived,
    **map_cc,
    **map_eu,
    **map_dataset
}

source_results_df["uniqueness_check"] = source_results_df["Code"].map(combined_map)
source_results_df["uniqueness_check"] = (
    source_results_df["uniqueness_check"]
    .astype("boolean")
    .fillna(True)
)

false_uniqueness_titles = source_results_df.loc[
    source_results_df["uniqueness_check"] == False,
    "Title EN"
]

condition = (
    source_results_df["Title EN"].isin(false_uniqueness_titles) &
    (source_results_df["Dissemination Product Type"] == "DATASET") &
    (source_results_df["uniqueness_check"] != False)
)

source_results_df.loc[condition, "uniqueness_check"] = False

# Final title score
check_columns = [
    "acronym_check",
    "breakdown_check",
    "length_check",
    "separators_check",
    "letters_check",
    "uniqueness_check",
    "spaces_check",
    "indicator_check",
    "index_check",
    "periodicity_check",
    "units_check"
]

for c in check_columns:
    source_results_df[c] = source_results_df[c].fillna(False).astype(bool)

source_results_df["dataset_title_standards_score"] = source_results_df[check_columns].mean(axis=1).round(4) * 100

messages = {
    "acronym_check": "The acronym was not found",
    "breakdown_check": "The breakdown is incorrect",
    "length_check": "The length exceeds 149 characters",
    "separators_check": "Parentheses are included more than once",
    "letters_check": "The first letter must be uppercase",
    "uniqueness_check": "This title has already been used",
    "spaces_check": "There are double spaces present",
    "indicator_check": "The indicator format is invalid",
    "index_check": "The index format is invalid",
    "periodicity_check": "The periodicity is not expressed correctly",
    "units_check": "The unit of measure is invalid"
}

source_results_df["dataset_title_standards_error"] = source_results_df.apply(
    lambda row: ", ".join([
        messages[c]
        for c in check_columns
        if not row[c]
    ]),
    axis=1
)

source_results_df["dataset_title_standards_error"] = source_results_df[
    "dataset_title_standards_error"
].apply(
    lambda x: "Dataset Title Standards Verification [" + x + "]" if x != "" else None
)

title_results_pd = source_results_df[
    [
        "Code",
        "dataset_title_standards_score",
        "dataset_title_standards_error"
    ]
].drop_duplicates(subset=["Code"]).rename(columns={
    "Code": "title_dataset_code"
})

title_results_pd["dataset_title_standards_score"] = title_results_pd[
    "dataset_title_standards_score"
].astype(float)

title_results_pd["dataset_title_standards_error"] = title_results_pd[
    "dataset_title_standards_error"
].astype("string")

title_schema = StructType([
    StructField("title_dataset_code", StringType(), True),
    StructField("dataset_title_standards_score", DoubleType(), True),
    StructField("dataset_title_standards_error", StringType(), True)
])

title_results_df = spark.createDataFrame(title_results_pd, schema=title_schema)

checks_df = checks_df.join(
    title_results_df,
    checks_df.dataset_code == title_results_df.title_dataset_code,
    "left"
).drop("title_dataset_code")

# =========================================================
# Validation of Historical Data Listing
# =========================================================

historical_df = source_results_df.copy()

# =========================================================
# Map dataStart / dataEnd from toc_df using Code
# =========================================================

historical_df["Code_Preprocessed"] = (
    historical_df["Code"]
    .astype(str)
    .str.split("$", regex=False)
    .str[0]
    .str.strip()
)

toc_tmp = toc_df.copy()

toc_tmp["Code"] = (
    toc_tmp["Code"]
    .astype(str)
    .str.strip()
)

code_to_datastart = (
    toc_tmp
    .set_index("Code")["dataStart"]
    .to_dict()
)

code_to_dataend = (
    toc_tmp
    .set_index("Code")["dataEnd"]
    .to_dict()
)

historical_df["dataStart_raw"] = historical_df["Code_Preprocessed"].map(code_to_datastart)
historical_df["dataEnd_raw"] = historical_df["Code_Preprocessed"].map(code_to_dataend)

# Extract year safely
historical_df["dataStart"] = (
    historical_df["dataStart_raw"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

historical_df["dataEnd"] = (
    historical_df["dataEnd_raw"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

historical_df["dataStart"] = pd.to_numeric(
    historical_df["dataStart"],
    errors="coerce"
)

historical_df["dataEnd"] = pd.to_numeric(
    historical_df["dataEnd"],
    errors="coerce"
)

if "Dissemination Product Type" not in historical_df.columns:
    historical_df["Dissemination Product Type"] = historical_df.get(
        "type",
        "DATASET"
    ).astype(str).str.upper()

historical_df["dataEnd_numeric"] = historical_df["dataEnd"]

historical_df["Last_Value_Less_Than_2014"] = historical_df["dataEnd_numeric"].apply(
    lambda x: "No" if pd.isna(x) else "Yes" if x < 2014 else "No"
)

historical_df["One_Reference_Year"] = historical_df.apply(
    lambda row: "Yes"
    if pd.notna(row["dataStart"]) and pd.notna(row["dataEnd"]) and row["dataStart"] == row["dataEnd"]
    else "No",
    axis=1
)

historical_df["Include_Historical"] = historical_df["Title EN"].str.contains(
    "historical",
    case=False,
    na=False
).map({True: "Yes", False: "No"})

def contains_year(title):
    title = "" if pd.isna(title) else str(title)
    year_pattern = r"\b(?:19|20)\d{2}\b"
    return "Yes" if re.search(year_pattern, title) else "No"

historical_df["Include_Year"] = historical_df["Title EN"].apply(contains_year)

historical_df["ends_with_h"] = historical_df["Code"].apply(
    lambda x: "Yes" if str(x).endswith("_h") else "No"
)

def check_historical_data_prefix(title):
    title = "" if pd.isna(title) else str(title).lower()

    if "historical" in title:
        idx = title.find("historical")
        return idx > 1 and title[idx - 2:idx] == "- "

    return True

historical_df["historical_check"] = historical_df["Title EN"].apply(
    check_historical_data_prefix
)

historical_df[
    [
        "Last_Value_Less_Than_2014",
        "Include_Historical",
        "Include_Year",
        "Dissemination Product Type",
        "One_Reference_Year",
        "ends_with_h"
    ]
] = historical_df[
    [
        "Last_Value_Less_Than_2014",
        "Include_Historical",
        "Include_Year",
        "Dissemination Product Type",
        "One_Reference_Year",
        "ends_with_h"
    ]
].fillna("")

historical_df["historical_check"] = (
    historical_df["historical_check"]
    .fillna(False)
    .astype(bool)
)

def check_historical_status(row):
    failure_messages = []

    if (
        row["Last_Value_Less_Than_2014"] == "Yes"
        and row["Include_Historical"] == "Yes"
        and row["Dissemination Product Type"] != "CATEGORY"
    ):
        failure_messages.append(
            "Last value less than 2014, Product is DATASET but historical is in the title"
        )

    if (
        row["Last_Value_Less_Than_2014"] == "Yes"
        and row["Include_Year"] == "No"
        and row["Dissemination Product Type"] != "CATEGORY"
    ):
        failure_messages.append(
            "Last value less than 2014, Product is DATASET but Year is not in the title"
        )

    if (
        (
            row["Last_Value_Less_Than_2014"] == "Yes"
            or row["One_Reference_Year"] == "Yes"
        )
        and row["Include_Year"] == "No"
        and row["Dissemination Product Type"] != "CATEGORY"
    ):
        failure_messages.append(
            "Last value less than 2014 or one reference Year, Product is DATASET but year is not in the title"
        )

    if (
        row["Include_Historical"] == "Yes"
        and row["Include_Year"] == "Yes"
        and row["Dissemination Product Type"] == "CATEGORY"
    ):
        failure_messages.append(
            "Historical included in the title, Product is CATEGORY but Year is in the title"
        )

    if (
        row["Include_Historical"] == "No"
        and row["ends_with_h"] == "Yes"
        and row["Dissemination Product Type"] == "CATEGORY"
    ):
        failure_messages.append(
            "Code ends with _h, Product is CATEGORY but historical is not in the title"
        )

    if row["historical_check"] is not True:
        failure_messages.append(
            "Historical is not correctly prefixed in the title"
        )

    if failure_messages:
        return "; ".join(failure_messages)

    return "Pass"

historical_df["historical_status"] = historical_df.apply(
    check_historical_status,
    axis=1
)

historical_df["historical_data_listing_score"] = historical_df["historical_status"].apply(
    lambda x: 100.0 if x == "Pass" else 0.0
)

historical_df["historical_data_listing_error"] = historical_df["historical_status"].apply(
    lambda x: None if x == "Pass"
    else "Validation of Historical Data Listing [" + x + "]"
)

historical_results_pd = historical_df[
    [
        "Code",
        "dataStart_raw",
        "dataEnd_raw",
        "dataStart",
        "dataEnd",
        "Last_Value_Less_Than_2014",
        "One_Reference_Year",
        "Include_Historical",
        "Include_Year",
        "ends_with_h",
        "historical_check",
        "historical_status",
        "historical_data_listing_score",
        "historical_data_listing_error"
    ]
].drop_duplicates(subset=["Code"]).rename(columns={
    "Code": "historical_dataset_code"
})

historical_results_pd["historical_data_listing_score"] = (
    historical_results_pd["historical_data_listing_score"]
    .astype(float)
)

historical_results_pd["historical_data_listing_error"] = (
    historical_results_pd["historical_data_listing_error"]
    .astype("string")
)

historical_schema = StructType([
    StructField("historical_dataset_code", StringType(), True),
    StructField("dataStart_raw", StringType(), True),
    StructField("dataEnd_raw", StringType(), True),
    StructField("dataStart", DoubleType(), True),
    StructField("dataEnd", DoubleType(), True),
    StructField("Last_Value_Less_Than_2014", StringType(), True),
    StructField("One_Reference_Year", StringType(), True),
    StructField("Include_Historical", StringType(), True),
    StructField("Include_Year", StringType(), True),
    StructField("ends_with_h", StringType(), True),
    StructField("historical_check", BooleanType(), True),
    StructField("historical_status", StringType(), True),
    StructField("historical_data_listing_score", DoubleType(), True),
    StructField("historical_data_listing_error", StringType(), True)
])

historical_results_df = spark.createDataFrame(
    historical_results_pd,
    schema=historical_schema
)

checks_df = checks_df.join(
    historical_results_df,
    checks_df.dataset_code == historical_results_df.historical_dataset_code,
    "left"
).drop("historical_dataset_code")

# =========================================================
# Dimensional Completeness
# =========================================================

dimensions_pd = spark.table("workspace.default.emqf_dimensions_metadata").toPandas()
dimensions_pd = dimensions_pd[dimensions_pd["Dataset_Code"].isin(dataset_codes)].copy()

dimensions_pd["CODELIST_CODE"] = dimensions_pd["CODELIST_CODE"].astype(str).str.lower()
dimensions_pd["DERIVED_CL"] = dimensions_pd["DERIVED_CL"].astype(str).str.lower()
dimensions_pd["STANDARD_CODE_SC"] = dimensions_pd["STANDARD_CODE_SC"].astype(str).str.upper()

geo_alternatives = [
    "geo", "airpol", "rep_airp", "airp_pr", "cities",
    "fishreg", "rbd", "port_iww", "par_mar",
    "metroreg", "net_seg10", "net_seg15", "net_seg"
]

def check_time_geo_unit(group):
    codes = set(group["CODELIST_CODE"])

    has_time = "time" in codes

    has_geo = (
        any(code in codes for code in geo_alternatives) or
        (group["DERIVED_CL"] == "geo").any() or
        (group["DERIVED_CL"].str.contains("geo", na=False)).any()
    )

    has_unit = (
        "unit" in codes or
        "currency" in codes or
        "statinfo" in codes or
        any(
            str(code).startswith("indic") and str(sc) in ["Y", "C"]
            for code, sc in zip(group["CODELIST_CODE"], group["STANDARD_CODE_SC"])
        )
    )

    found = sum([has_time, has_geo, has_unit])
    score = round((found / 3) * 100, 2)

    missing = []
    if not has_time:
        missing.append("TIME")
    if not has_geo:
        missing.append("GEO")
    if not has_unit:
        missing.append("UNIT")

    return pd.Series({
        "has_time": bool(has_time),
        "has_geo": bool(has_geo),
        "has_unit": bool(has_unit),
        "dimensional_completeness_score": float(score),
        "dimensional_completeness_error": "Missing required dimension group(s): " + ", ".join(missing) if missing else None
    })

if dimensions_pd.empty:
    dimensional_pd = pd.DataFrame({
        "dim_dataset_code": dataset_codes,
        "has_time": False,
        "has_geo": False,
        "has_unit": False,
        "dimensional_completeness_score": 0.0,
        "dimensional_completeness_error": "No dimensions metadata available"
    })
else:
    dimensional_pd = (
        dimensions_pd
        .groupby("Dataset_Code", group_keys=False)
        .apply(check_time_geo_unit)
        .reset_index()
        .rename(columns={"Dataset_Code": "dim_dataset_code"})
    )

dim_schema = StructType([
    StructField("dim_dataset_code", StringType(), True),
    StructField("has_time", BooleanType(), True),
    StructField("has_geo", BooleanType(), True),
    StructField("has_unit", BooleanType(), True),
    StructField("dimensional_completeness_score", DoubleType(), True),
    StructField("dimensional_completeness_error", StringType(), True)
])

dimensional_spark_df = spark.createDataFrame(dimensional_pd, schema=dim_schema)

checks_df = checks_df.join(
    dimensional_spark_df,
    checks_df.dataset_code == dimensional_spark_df.dim_dataset_code,
    "left"
).drop("dim_dataset_code")

# =========================================================
# Standard Codelists Percentage
# =========================================================

def standard_codelist_percentage(group):
    values = group["STANDARD_CODE_SC"].dropna().astype(str).str.upper().str.strip()
    values = values[~values.isin(["", "NAN", "NONE", "EMPTY"])]

    total = len(values)

    if total == 0:
        return pd.Series({
            "standard_codelists_total": 0,
            "standard_codelists_standard_count": 0,
            "standard_codelists_non_standard_count": 0,
            "standard_codelists_score": 0.0,
            "standard_codelists_error": "No codelist standard code information available"
        })

    standard_count = int(values.isin(["Y", "C"]).sum())
    non_standard_count = int(total - standard_count)
    score = round((standard_count / total) * 100, 2)

    return pd.Series({
        "standard_codelists_total": int(total),
        "standard_codelists_standard_count": standard_count,
        "standard_codelists_non_standard_count": non_standard_count,
        "standard_codelists_score": float(score),
        "standard_codelists_error": f"{non_standard_count} non-standard codelist(s) found" if non_standard_count > 0 else None
    })

if dimensions_pd.empty:
    standard_pd = pd.DataFrame({
        "std_dataset_code": dataset_codes,
        "standard_codelists_total": 0,
        "standard_codelists_standard_count": 0,
        "standard_codelists_non_standard_count": 0,
        "standard_codelists_score": 0.0,
        "standard_codelists_error": "No codelist standard code information available"
    })
else:
    standard_pd = (
        dimensions_pd
        .groupby("Dataset_Code", group_keys=False)
        .apply(standard_codelist_percentage)
        .reset_index()
        .rename(columns={"Dataset_Code": "std_dataset_code"})
    )

standard_pd["standard_codelists_error"] = standard_pd["standard_codelists_error"].astype("string")

std_schema = StructType([
    StructField("std_dataset_code", StringType(), True),
    StructField("standard_codelists_total", IntegerType(), True),
    StructField("standard_codelists_standard_count", IntegerType(), True),
    StructField("standard_codelists_non_standard_count", IntegerType(), True),
    StructField("standard_codelists_score", DoubleType(), True),
    StructField("standard_codelists_error", StringType(), True)
])

standard_spark_df = spark.createDataFrame(standard_pd, schema=std_schema)

checks_df = checks_df.join(
    standard_spark_df,
    checks_df.dataset_code == standard_spark_df.std_dataset_code,
    "left"
).drop("std_dataset_code")

# =========================================================
# Helper: get GEO values without eurostat package
# =========================================================

def get_geo_values(dataset_code):

    try:
        url = (
            "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/"
            f"data/{dataset_code}"
        )

        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return []

        data = response.json()

        if "dimension" not in data:
            return []

        if "geo" not in data["dimension"]:
            return []

        geo_index = (
            data["dimension"]
            ["geo"]
            ["category"]
            ["index"]
        )

        return list(geo_index.keys())

    except Exception:
        return []

# =========================================================
# CACHE GEO VALUES ONCE
# =========================================================

import concurrent.futures

geo_values_cache = {}

valid_dataset_codes = [
    code for code in dataset_codes
    if code not in [None, "None", "nan", ""]
]

with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:

    futures = {
        executor.submit(get_geo_values, code): code
        for code in valid_dataset_codes
    }

    for future in concurrent.futures.as_completed(futures):

        code = futures[future]

        try:
            geo_values_cache[code] = future.result()

        except Exception:
            geo_values_cache[code] = []

# =========================================================
# EA20 Aggregate Consistency Check
# =========================================================

import urllib.request
import gzip
from io import BytesIO

toc_codes = dataset_codes

candidate_codes = [
    code
    for code, geo_values in geo_values_cache.items()
    if "EA19" in geo_values and "EA20" in geo_values
]

aggregates_df = pd.DataFrame({
    "Dataset_Code": candidate_codes
}).drop_duplicates(subset=["Dataset_Code"])

if aggregates_df.empty:

    aggregates_df = pd.DataFrame({
        "Dataset_Code": toc_codes,
        "ea20_has_flag": None,
        "ea20_non_positive_differences": None,
        "ea20_aggregate_consistency_score": 100.0,
        "ea20_aggregate_consistency_error": None
    })

else:

    aggregates_df["ea20_has_flag"] = None
    aggregates_df["ea20_non_positive_differences"] = None

    for index, row in aggregates_df.iterrows():

        dataset_code_value = row["Dataset_Code"]

        url = (
            "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
            f"data/dataflow/ESTAT/{dataset_code_value}/1.0"
            "?compress=true&format=TSV&c[GEO]=EA20,EA19"
        )

        try:

            response = urllib.request.urlopen(url, timeout=20)

            compressed_data = response.read()

            with gzip.GzipFile(fileobj=BytesIO(compressed_data)) as decompressed_file:
                data = pd.read_csv(decompressed_file, sep="\t")

            data = data.loc[:, ~(data == ": ").all()]
            data = data.replace(": ", 0)

            first_column_split = data.iloc[:, 0].astype(str).str.split(",", expand=True)
            first_column_header_split = data.columns[0].split(",")

            if len(first_column_header_split) != first_column_split.shape[1]:

                first_column_new_names = [
                    f"Col{i+1}"
                    for i in range(first_column_split.shape[1])
                ]

            else:

                first_column_new_names = first_column_header_split

            first_column_split.columns = first_column_new_names

            data = pd.concat(
                [first_column_split, data.iloc[:, 1:]],
                axis=1
            )

            data.columns = (
                list(first_column_new_names) +
                list(data.columns[len(first_column_new_names):])
            )

            geo_col = None

            for c in data.columns:

                if "geo" in c.lower():
                    geo_col = c
                    break

            if geo_col is None:

                aggregates_df.at[index, "ea20_has_flag"] = None
                aggregates_df.at[index, "ea20_non_positive_differences"] = None

                continue

            ea19_rows = data[
                data[geo_col].astype(str) == "EA19"
            ].reset_index(drop=True)

            ea20_rows_data = data[
                data[geo_col].astype(str) == "EA20"
            ].reset_index(drop=True)

            flags_exist = data.astype(str).apply(
                lambda x: x.str.contains(r"[a-z]", regex=True).any()
            ).any()

            aggregates_df.at[index, "ea20_has_flag"] = bool(flags_exist)

            if ea19_rows.empty or ea20_rows_data.empty:

                aggregates_df.at[index, "ea20_non_positive_differences"] = None

                continue

            geo_index = data.columns.get_loc(geo_col)

            non_positive = False

            for col_name in data.columns[geo_index + 1:]:

                ea19_value = pd.to_numeric(
                    ea19_rows[col_name].values[0],
                    errors="coerce"
                )

                ea20_value = pd.to_numeric(
                    ea20_rows_data[col_name].values[0],
                    errors="coerce"
                )

                if pd.notna(ea19_value) and pd.notna(ea20_value):

                    if ea20_value - ea19_value <= 0:
                        non_positive = True
                        break

            aggregates_df.at[index, "ea20_non_positive_differences"] = bool(non_positive)

        except Exception:

            aggregates_df.at[index, "ea20_has_flag"] = None
            aggregates_df.at[index, "ea20_non_positive_differences"] = None

    missing_codes = [
        code for code in toc_codes
        if code not in aggregates_df["Dataset_Code"].tolist()
    ]

    missing_df = pd.DataFrame({
        "Dataset_Code": missing_codes,
        "ea20_has_flag": None,
        "ea20_non_positive_differences": None
    })

    aggregates_df = pd.concat(
        [aggregates_df, missing_df],
        ignore_index=True
    )

    aggregates_df["ea20_aggregate_consistency_score"] = (
        aggregates_df["ea20_non_positive_differences"]
        .apply(lambda x: 0.0 if x is True else 100.0)
    )

    aggregates_df["ea20_aggregate_consistency_error"] = (
        aggregates_df["ea20_aggregate_consistency_score"]
        .apply(
            lambda x:
            "EA20 aggregate consistency check [EA20 aggregate inconsistency]"
            if x == 0.0 else None
        )
    )

# =========================================================
# Join EA20 results back to checks_df
# =========================================================

ea20_results_pd = aggregates_df[
    [
        "Dataset_Code",
        "ea20_has_flag",
        "ea20_non_positive_differences",
        "ea20_aggregate_consistency_score",
        "ea20_aggregate_consistency_error"
    ]
].drop_duplicates(subset=["Dataset_Code"]).rename(columns={
    "Dataset_Code": "ea20_dataset_code"
})

ea20_results_pd["ea20_has_flag"] = (
    ea20_results_pd["ea20_has_flag"]
    .astype("string")
)

ea20_results_pd["ea20_non_positive_differences"] = (
    ea20_results_pd["ea20_non_positive_differences"]
    .astype("string")
)

ea20_results_pd["ea20_aggregate_consistency_score"] = (
    ea20_results_pd["ea20_aggregate_consistency_score"]
    .astype(float)
)

ea20_results_pd["ea20_aggregate_consistency_error"] = (
    ea20_results_pd["ea20_aggregate_consistency_error"]
    .astype("string")
)

ea20_schema = StructType([
    StructField("ea20_dataset_code", StringType(), True),
    StructField("ea20_has_flag", StringType(), True),
    StructField("ea20_non_positive_differences", StringType(), True),
    StructField("ea20_aggregate_consistency_score", DoubleType(), True),
    StructField("ea20_aggregate_consistency_error", StringType(), True)
])

ea20_results_df = spark.createDataFrame(
    ea20_results_pd,
    schema=ea20_schema
)

checks_df = checks_df.join(
    ea20_results_df,
    checks_df.dataset_code == ea20_results_df.ea20_dataset_code,
    "left"
).drop("ea20_dataset_code")

# =========================================================
# EU27_2020 Aggregate Confidentiality Check
# =========================================================

eu27_candidate_codes = [
    code
    for code, geo_values in geo_values_cache.items()
    if "EU27_2020" in geo_values
]

aggregates_conf_df = pd.DataFrame({
    "Dataset_Code": eu27_candidate_codes
}).drop_duplicates(subset=["Dataset_Code"])

dataframes_conf = {}

if aggregates_conf_df.empty:
    aggregates_conf_df = pd.DataFrame({
        "Dataset_Code": dataset_codes,
        "eu27_has_flag": None,
        "eu27_has_nace": None,
        "eu27_confidentiality_check": True,
        "eu27_confidentiality_score": 100.0,
        "eu27_confidentiality_error": None
    })

else:
    aggregates_conf_df["eu27_has_flag"] = False
    aggregates_conf_df["eu27_has_nace"] = False
    aggregates_conf_df["eu27_confidentiality_check"] = True

    for index, row in aggregates_conf_df.iterrows():
        dataset_code_value = row["Dataset_Code"]

        url = (
            "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/"
            f"data/dataflow/ESTAT/{dataset_code_value}/1.0"
            "?compress=true&format=TSV&c[GEO]=EU27_2020"
        )

        try:
            response = urllib.request.urlopen(url, timeout=60)
            compressed_data = response.read()

            with gzip.GzipFile(fileobj=BytesIO(compressed_data)) as decompressed_file:
                data = pd.read_csv(decompressed_file, sep="\t")

            data = data.loc[:, ~(data == ": ").all()]

            flags_exist = data.astype(str).apply(
                lambda x: x.str.contains(r"c", regex=True).any()
            ).any()

            aggregates_conf_df.at[index, "eu27_has_flag"] = bool(flags_exist)

            first_column_split = data.iloc[:, 0].astype(str).str.split(",", expand=True)
            first_column_header_split = data.columns[0].split(",")

            if len(first_column_header_split) != first_column_split.shape[1]:
                first_column_new_names = [
                    f"Col{i+1}"
                    for i in range(first_column_split.shape[1])
                ]
            else:
                first_column_new_names = first_column_header_split

            first_column_split.columns = first_column_new_names

            data = pd.concat(
                [first_column_split, data.iloc[:, 1:]],
                axis=1
            )

            data.columns = list(first_column_new_names) + list(data.columns[len(first_column_new_names):])
            data.columns = data.columns.str.strip()

            has_nace = "nace_r2" in data.columns
            aggregates_conf_df.at[index, "eu27_has_nace"] = bool(has_nace)

            dataframes_conf[dataset_code_value] = data

        except Exception:
            aggregates_conf_df.at[index, "eu27_has_flag"] = None
            aggregates_conf_df.at[index, "eu27_has_nace"] = None
            aggregates_conf_df.at[index, "eu27_confidentiality_check"] = True

    valid_rows = aggregates_conf_df.loc[
        (aggregates_conf_df["eu27_has_flag"] == True) &
        (aggregates_conf_df["eu27_has_nace"] == True),
        "Dataset_Code"
    ]

    def process_nace_columns(df):
        df["Valid_Nace"] = df["nace_r2"].apply(
            lambda x: bool(re.match(r"^[A-Z]\d*$", x)) if isinstance(x, str) else False
        )

        df.loc[df["Valid_Nace"], "nace_digits"] = df["nace_r2"].apply(
            lambda x: len(re.findall(r"\d", x)) if isinstance(x, str) else 0
        )

        df.loc[df["Valid_Nace"], "Section"] = df["nace_r2"].apply(
            lambda x: x[0] if isinstance(x, str) else ""
        )

        df.loc[df["Valid_Nace"], "Division"] = df.apply(
            lambda row: row["nace_r2"][1:3] if row["Valid_Nace"] and row["nace_digits"] > 0 else "",
            axis=1
        )

        df.loc[df["Valid_Nace"], "Group"] = df.apply(
            lambda row: row["nace_r2"][1:4] if row["Valid_Nace"] and row["nace_digits"] >= 3 else "",
            axis=1
        )

        df.loc[df["Valid_Nace"], "Class"] = df.apply(
            lambda row: row["nace_r2"][1:] if row["Valid_Nace"] and row["nace_digits"] >= 4 else "",
            axis=1
        )

        return df

    for dataset_code in valid_rows:
        dataframes_conf[dataset_code] = process_nace_columns(dataframes_conf[dataset_code])

    for dataset_code in valid_rows:
        df = dataframes_conf[dataset_code]
        df.columns = df.columns.str.strip()

        failures = []

        if "geo\\TIME_PERIOD" not in df.columns or "Section" not in df.columns:
            aggregates_conf_df.loc[
                aggregates_conf_df["Dataset_Code"] == dataset_code,
                "eu27_confidentiality_check"
            ] = True
            continue

        geo_index = df.columns.get_loc("geo\\TIME_PERIOD")
        section_index = df.columns.get_loc("Section")

        columns_to_check = df.columns[geo_index + 1:section_index]
        columns_before_geo = df.columns[:geo_index].difference(["nace_r2"]).tolist()

        valid_nace_df = df[df["Valid_Nace"] == True].copy()

        for col_name in columns_to_check:
            flagged_rows = valid_nace_df[valid_nace_df[col_name] == ": c"].copy()

            if flagged_rows.empty:
                continue

            for idx, flagged_row in flagged_rows.iterrows():
                nace_digits = flagged_row["nace_digits"]

                if nace_digits < 3:
                    continue

                potential_matches = valid_nace_df[
                    (valid_nace_df[columns_before_geo] == flagged_row[columns_before_geo]).all(axis=1)
                ]

                if nace_digits == 3:
                    broader_matches = potential_matches[
                        (potential_matches["Division"] == flagged_row["Division"]) &
                        (potential_matches["nace_digits"] == 2)
                    ]

                    sibling_matches = potential_matches[
                        (potential_matches["Division"] == flagged_row["Division"]) &
                        (potential_matches["nace_digits"] == 3)
                    ]

                elif nace_digits == 4:
                    broader_matches = potential_matches[
                        (potential_matches["Group"] == flagged_row["Group"]) &
                        (potential_matches["nace_digits"] == 3)
                    ]

                    sibling_matches = potential_matches[
                        (potential_matches["Group"] == flagged_row["Group"]) &
                        (potential_matches["nace_digits"] == 4)
                    ]

                else:
                    continue

                combined_matches = pd.concat([sibling_matches, broader_matches])

                if combined_matches.empty or len(combined_matches) == 1:
                    continue

                if combined_matches["nace_digits"].nunique() == 1:
                    continue

                if len(combined_matches) == 2:
                    if (
                        (combined_matches[col_name].astype(str).str.contains(": c").sum() == 1) and
                        (combined_matches[col_name].astype(str).str.contains(r"\d", regex=True).sum() == 1)
                    ):
                        continue

                num_confidential_matches = combined_matches[col_name].isin([": c"]).sum()
                has_valid_value = combined_matches[col_name].astype(str).str.contains(r"\d", regex=True).any()

                if num_confidential_matches <= 1 and has_valid_value:
                    failures.append({
                        "Dataset_Code": dataset_code,
                        "Column": col_name,
                        "Initial_Row_Index": idx,
                        "Compared_Row_Indexes": combined_matches.index.tolist()
                    })

        aggregates_conf_df.loc[
            aggregates_conf_df["Dataset_Code"] == dataset_code,
            "eu27_confidentiality_check"
        ] = len(failures) == 0

    missing_codes = [
        code for code in dataset_codes
        if code not in aggregates_conf_df["Dataset_Code"].tolist()
    ]

    missing_df = pd.DataFrame({
        "Dataset_Code": missing_codes,
        "eu27_has_flag": None,
        "eu27_has_nace": None,
        "eu27_confidentiality_check": True
    })

    aggregates_conf_df = pd.concat(
        [aggregates_conf_df, missing_df],
        ignore_index=True
    )

    aggregates_conf_df["eu27_confidentiality_score"] = aggregates_conf_df[
        "eu27_confidentiality_check"
    ].apply(
        lambda x: 100.0 if x is True else 0.0
    )

    aggregates_conf_df["eu27_confidentiality_error"] = aggregates_conf_df[
        "eu27_confidentiality_check"
    ].apply(
        lambda x: None if x is True
        else "EU27 aggregate confidential inconsistency [EU27 aggregate confidential inconsistency]"
    )

eu27_results_pd = aggregates_conf_df[
    [
        "Dataset_Code",
        "eu27_has_flag",
        "eu27_has_nace",
        "eu27_confidentiality_check",
        "eu27_confidentiality_score",
        "eu27_confidentiality_error"
    ]
].drop_duplicates(subset=["Dataset_Code"]).rename(columns={
    "Dataset_Code": "eu27_dataset_code"
})

eu27_results_pd["eu27_has_flag"] = eu27_results_pd["eu27_has_flag"].astype("string")
eu27_results_pd["eu27_has_nace"] = eu27_results_pd["eu27_has_nace"].astype("string")
eu27_results_pd["eu27_confidentiality_check"] = eu27_results_pd["eu27_confidentiality_check"].astype("string")
eu27_results_pd["eu27_confidentiality_score"] = eu27_results_pd["eu27_confidentiality_score"].astype(float)
eu27_results_pd["eu27_confidentiality_error"] = eu27_results_pd["eu27_confidentiality_error"].astype("string")

eu27_schema = StructType([
    StructField("eu27_dataset_code", StringType(), True),
    StructField("eu27_has_flag", StringType(), True),
    StructField("eu27_has_nace", StringType(), True),
    StructField("eu27_confidentiality_check", StringType(), True),
    StructField("eu27_confidentiality_score", DoubleType(), True),
    StructField("eu27_confidentiality_error", StringType(), True)
])

eu27_results_df = spark.createDataFrame(
    eu27_results_pd,
    schema=eu27_schema
)

checks_df = checks_df.join(
    eu27_results_df,
    checks_df.dataset_code == eu27_results_df.eu27_dataset_code,
    "left"
).drop("eu27_dataset_code")

# =========================================================
# Fill nulls + write final daily checks table
# =========================================================

checks_df = checks_df.fillna({
    "data_browser_verification_score": 0.0,
    "source_of_data_validation_score": 0.0,
    "dimensional_completeness_score": 0.0,
    "standard_codelists_total": 0,
    "standard_codelists_standard_count": 0,
    "standard_codelists_non_standard_count": 0,
    "standard_codelists_score": 0.0,
    "ea20_aggregate_consistency_score": 100.0,
    "eu27_confidentiality_score": 100.0
})

checks_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_daily_quality_checks")

display(checks_df)

## 9. Final score table

In [0]:
from functools import reduce
from datetime import datetime

import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, coalesce, when, trim

checks_df = spark.table("workspace.default.emqf_daily_quality_checks")

score_columns = [
    "metadata_file_existence_esms_score",
    "doi_verification_score",
    "op_dataset_availability_score",
    "dataset_title_standards_score",
    "source_of_data_validation_score",
    "historical_data_listing_score",
    "data_browser_verification_score",
    "dimensional_completeness_score",
    "standard_codelists_score",
    "ea20_aggregate_consistency_score",
    "eu27_confidentiality_score"
]

existing_score_columns = [
    c for c in score_columns
    if c in checks_df.columns
]

for c in existing_score_columns:
    checks_df = checks_df.withColumn(
        c,
        coalesce(col(c).cast("double"), lit(0.0))
    )

score_sum = reduce(
    lambda a, b: a + b,
    [col(c) for c in existing_score_columns]
)

error_columns = [
    c.replace("_score", "_error")
    for c in existing_score_columns
    if c.replace("_score", "_error") in checks_df.columns
]

for c in error_columns:
    checks_df = checks_df.withColumn(
        c,
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (trim(col(c).cast("string")).isin("None", "none", "nan", "NaN", "<NA>")),
            lit(None).cast("string")
        ).otherwise(col(c).cast("string"))
    )

final_df = checks_df.withColumn(
    "final_quality_score",
    F.round(score_sum / lit(len(existing_score_columns)), 2)
).withColumn(
    "failed_conditions",
    F.concat_ws(
        "; ",
        *[col(c) for c in error_columns]
    )
).withColumn(
    "failed_conditions",
    when(
        trim(col("failed_conditions")) == "",
        lit(None).cast("string")
    ).otherwise(col("failed_conditions"))
).withColumn(
    "run_timestamp",
    F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
).withColumn(
    "run_date",
    F.current_date()
)

final_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.emqf_daily_quality_score_history")

final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.emqf_daily_quality_score_latest")

display(final_df.select(
    "run_date",
    "run_timestamp",
    "dataset_code",
    "title",
    *existing_score_columns,
    "final_quality_score",
    "failed_conditions"
))

## 10. SQL analytics

In [0]:
%sql
SELECT*
FROM workspace.default.emqf_daily_quality_score_history
ORDER BY final_quality_score ASC;

In [0]:
%sql
SELECT
    dataset_code,
    substring(title, 1, 60) AS title,
    final_quality_score,
    failed_conditions,
    run_date
FROM workspace.default.emqf_daily_quality_score_history
LIMIT 15;

In [0]:
%sql
SELECT
    dataset_code,
    substring(title, 1, 60) AS title,
    final_quality_score,
    CASE
        WHEN failed_conditions IS NULL THEN ''
        ELSE failed_conditions
    END AS failed_conditions,
    run_date
FROM workspace.default.emqf_daily_quality_score_history
LIMIT 15;